# 🤖 Phase 2: GenAI Frameworks & Building AI Agents
### Topics: Tool Calling | Agentic Workflows | Decision Making | Research Agent

---
> **Note for students**: Run each cell **in order** from top to bottom. Never skip cells!

## 📚 What You Will Learn
| Section | Topic |
|---------|-------|
| Step 0 | Installation |
| Step 1 | Setup API Key |
| Part 1 | What is an AI Agent? |
| Part 2 | Tool Calling — Binding Python Functions to LLMs |
| Part 3 | Agentic Workflows & Decision Making |
| Part 4 | Building a Real-Time Research Agent |

## ⬇️ Step 0: Install Libraries

We need the following packages:
- `langchain` — AI orchestration framework
- `langchain-groq` — Groq integration for LangChain
- `ddgs` — DuckDuckGo search library
- `langchain-community` — Community tools
- `langgraph` — For building complex agent workflows

In [ ]:
# Run this cell only ONCE to install all required libraries
!pip install -U langchain langchain-groq ddgs langchain-community langgraph python-dotenv

## 🔐 Step 1: Setup API Key

**⚠️ IMPORTANT**: Paste your Groq API key in the cell below.
Get a FREE key at: https://console.groq.com

In [ ]:
import os
from getpass import getpass

# Enter your key here! 
GROQ_API_KEY = "" # ← You can paste it here directly, or use the input box below

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ API Key set successfully!")

In [ ]:
from langchain_groq import ChatGroq

# Initialize the LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",   # Fast and free model
    temperature=0                 # 0 = focused/deterministic
)

print("🤖 LLM Ready:", llm.model_name)

---
# 🧠 PART 1: What Is an AI Agent?

An **AI Agent** is an LLM that can **use tools** to solve problems. 

While a chatbot just talks, an agent can:
1. **Analyze** the user's request.
2. **Choose** a tool (like Google Search or a Calculator).
3. **Execute** the tool and read the result.
4. **Repeat** until the goal is met.

In [ ]:
# Example of a regular LLM failing at real-time tasks
try:
    response = llm.invoke("What is the current price of Gold today?")
    print("🤖 LLM Answer:", response.content)
except Exception as e:
    print("Error:", e)

---
# 🔧 PART 2: Tool Calling

Tool calling allows us to connect Python functions to the LLM.

In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: float, b: float) -> float:
    """Adds two numbers together. Use this for math calculations."""
    return a + b

@tool
def get_word_count(text: str) -> int:
    """Counts words in a string."""
    return len(text.split())

tools = [add_numbers, get_word_count]
llm_with_tools = llm.bind_tools(tools)

print("✅ Tools bound to LLM!")

In [ ]:
# Ask a question that requires a tool
response = llm_with_tools.invoke("What is 1234.56 plus 789.12?")

print("🛠️ Tool calls requested:", response.tool_calls)

if response.tool_calls:
    tc = response.tool_calls[0]
    # Note: We still need to manually run the tool if not using an agent loop
    result = add_numbers.invoke(tc['args'])
    print("✅ Result from Python function:", result)

---
# 🔄 PART 3: Agentic Workflows & Decision Making

An agent uses a loop to decide which tool to call next.

In [ ]:
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain_core.messages import ToolMessage

def run_simple_agent(question):
    messages = [SystemMessage(content="You are helpful."), HumanMessage(content=question)]
    
    # 1. LLM decides
    response = llm_with_tools.invoke(messages)
    
    if not response.tool_calls:
        return response.content
    
    # 2. Executing tool
    messages.append(response)
    for tc in response.tool_calls:
        if tc['name'] == 'add_numbers':
            res = add_numbers.invoke(tc['args'])
            messages.append(ToolMessage(content=str(res), tool_call_id=tc['id']))
    
    # 3. LLM summarizes
    final_response = llm_with_tools.invoke(messages)
    return final_response.content

print("🤖 Agent says:", run_simple_agent("What is 50 + 50?"))

## 🚀 LangGraph ReAct Agent
LangGraph makes it much easier to build these loops.

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools)
resp = agent.invoke({"messages": [HumanMessage(content="Find out the word count of 'I love building AI agents'")]})

print("🤖 Final Answer:", resp['messages'][-1].content)

---
# 🌐 PART 4: Internet Research Agent

Now let's build an agent that can browse the web.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

research_agent = create_react_agent(llm, [search_tool])

question = "What are the top 3 AI trends in 2025?"
print(f"🔍 Researching: {question}...")

response = research_agent.invoke({"messages": [HumanMessage(content=question)]})
print("\n📋 Research Report:")
print(response['messages'][-1].content)

## ✍️ Final Challenge
Try asking the agent to research something specific you are interested in!

In [ ]:
user_topic = input("Enter a research topic: ")
if user_topic:
    res = research_agent.invoke({"messages": [HumanMessage(content=user_topic)]})
    print(res['messages'][-1].content)